# **Durability emulator — PCE training**

This notebook **only** fits and validates the PCE. It reads the `dataset_unique_train` / `dataset_unique_val` files written by [`01_generate_dataset.ipynb`](01_generate_dataset.ipynb).

## **1. Libraries**

In [ ]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import dill
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from functions_final import *
from UQpy.distributions import Uniform, JointIndependent

## **2. Random variables and fixed parameters**

Must match [`01_generate_dataset.ipynb`](01_generate_dataset.ipynb). This only rebuilds the
distribution object and the filename tag, it draws no new samples.

In [ ]:
fck_min = 20
fck_max = 50
rh_min  = 20
rh_max  = 80
cov_min = 15
cov_max = 60

cement_type         = 3
installation_year   = 1990
exposure_conditions = 2
n_latent_samples     = 100000   # must match stage 1 — it is the filename prefix
n_lambdas            = 4
max_degree           = 3        # maximum total degree of the PCE polynomial basis

fck_dist = Uniform(loc=fck_min, scale=fck_max - fck_min)
rh_dist  = Uniform(loc=rh_min, scale=rh_max - rh_min)
cov_dist = Uniform(loc=cov_min, scale=cov_max - cov_min)
joint    = JointIndependent(marginals=[fck_dist, rh_dist, cov_dist])

## **3. Time grid**

Must match times written by [`01_generate_dataset.ipynb`](01_generate_dataset.ipynb).

In [ ]:
times = np.linspace(0, 150, 10, endpoint=True)
times

## **4. Load the datasets and train the PCE at each time step**

In [ ]:
print("="*60)
print("TRAINING THE DURABILITY PCE")
print("="*60)

results = []
for t in times:
    tag = f'{t}_install_{installation_year}_cement_{cement_type}_exposure_{exposure_conditions}'
    with open(f'{n_latent_samples}_dataset_unique_train_{tag}.pkl', 'rb') as f:
        df_unique_train = dill.load(f)
    with open(f'{n_latent_samples}_dataset_unique_val_{tag}.pkl', 'rb') as f:
        df_unique_val = dill.load(f)

    result = train_and_validate_pce_from_dataset_durability(
                                                               df_unique_train=df_unique_train,
                                                               df_unique_val=df_unique_val,
                                                               joint=joint,
                                                               time_step=t,
                                                               installation_year=installation_year,
                                                               cement_type=cement_type,
                                                               exposure_conditions=exposure_conditions,
                                                               n_latent_samples=n_latent_samples,
                                                               n_lambdas=n_lambdas,
                                                               max_degree=max_degree,
                                                               output_dir='.',
                                                           )
    result['x_train'] = df_unique_train[['fck', 'rh', 'cov']].to_numpy()
    results.append(result)

## 5. Validation summary

How well the PCE reproduces each lambda, per time step.

In [ ]:
validation_summary = pd.concat([r['statistics'] for r in results], ignore_index=True)
validation_summary.insert(0, 'Time (years)', [r['time_step'] for r in results])
validation_summary

## 6. Emulator efficiency (speed-up)

Combines this notebook's PCE evaluation time with the emulator cost recorded by
[`01_generate_dataset.ipynb`](01_generate_dataset.ipynb).

In [ ]:
with open(f'{n_latent_samples}_emulator_timing_durability.pkl', 'rb') as f:
    emulator_timing = dill.load(f)

speedup_rows = []
for result in results:
    emulator_s = float(emulator_timing.loc[emulator_timing['Time (years)'] == result['time_step'], 'Train total (s)'].iloc[0])

    t_start = time.perf_counter()
    result['pce_metamodel'].predict(result['x_train'])
    surrogate_s = time.perf_counter() - t_start

    speedup_rows.append({
                            'Time (years)':  result['time_step'],
                            'Emulator (s)':  emulator_s,
                            'Surrogate (s)': surrogate_s,
                            'Speed-up':      emulator_s / surrogate_s,
                        })

speedup = pd.DataFrame(speedup_rows)
print(f"Median speed-up: {speedup['Speed-up'].median():,.0f}x")
speedup

### 6.1 Speed-up over time

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(speedup['Time (years)'], speedup['Speed-up'], marker='o', color='0.25')
ax.set_xlabel('Time (years)')
ax.set_ylabel('Speed-up (emulator / surrogate)')
ax.set_yscale('log')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f'{v:,.0f}x'))
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()